*Notebook 05 in `notebooks/arcs/`. Parameters (pre-registered): `analysis/arcs/config.yaml`; decisions and run notes: `analysis/arcs/LOG.md`. The static page `site/arcs/index.html` is built from the same saved outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/arcs -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.arcs import s05_arcs as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 5 · Arcs

**What we do.** For each script we take its scene scores in story order: the "both" layer, one score per
scene on the timeline. Then:
1. We put them on a common clock of **100 points of story time**. Scene *i* of *n* covers
   [*i*/*n*, (*i*+1)/*n*) of the story. Each grid point takes the mean of the scenes it overlaps, weighted by
   the overlap. This works the same for a script with 40 scenes as for one with 400; nothing is dropped or
   interpolated.
2. We **smooth** with a centred moving average over 11 points (about a tenth of the story), cut short at the
   two ends.
3. We summarise each arc with the two pre-registered **primary descriptors**:
   - **D1, volatility**: the standard deviation of the smoothed arc. How much the tone swings.
   - **D2, ending minus start**: the mean of the last 20% minus the mean of the first 20% of the unsmoothed
     arc. Does the story end lighter or darker than it began.

   Three exploratory descriptors come alongside (mean level, position of the low point, number of swings).

**Two checks.**
- **Proxies.** Are the descriptors just script size in disguise? A scene made of two sentences gives a noisy
  mean, so scripts of short scenes could look volatile for no narrative reason.
- **Reliability.** Each script is split into odd and even scenes, and every descriptor is computed on each
  half. If the halves disagree, scene noise dominates.

The instrument is the one the pre-registered fallback rule chose in Step 3 (named in every figure and reading);
the other instruments appear in Step 9. No outcome is used in this step.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

/Users/gyula/Documents/Projects/code/IMDb-movie-success-arcs/analysis/arcs/s05_arcs.py:36: RuntimeWarning: invalid value encountered in divide
  a = np.arange(n + 1) / n


**Figure.** Mean arc over the primary sample. Grey: smoothed. Blue: resampled, unsmoothed. Bands: ±1.96 standard errors across films.

In [ ]:
step.figures(res)["mean"].show()

**Figure.** Mean smoothed arc per genre bin.

In [ ]:
step.figures(res)["genres"].show()

**Table.** Median descriptors per genre bin (primary sample).

In [ ]:
display(step.tables(res)["genres"])

,primary_genre,films,D1,D2,mean_level
0,Action,66,0.1119,0.0218,-0.0672
1,Adventure,113,0.1268,0.0144,0.0176
2,Comedy,175,0.1301,0.0354,0.0105
3,Drama,203,0.1258,-0.0035,0.0036
4,Horror,98,0.1308,-0.0139,-0.0633
5,Other,4,0.1233,-0.0635,-0.0518
6,Sci-Fi,71,0.1201,0.0280,-0.0295
7,Thriller,104,0.1229,0.0228,-0.0646


**Figure.** Six arcs drawn with the config seed: numbers only, no script text.

In [ ]:
step.figures(res)["examples"].show()

**Table.** The six example scripts and their descriptors.

In [ ]:
display(step.tables(res)["examples"])

,movie_name,primary_genre,scenes,D1,D2,mean_level
6,127 Hours_2010,Drama,231,0.1514,0.2372,-0.0975
111,Barbarian_2022,Horror,64,0.1521,-0.0909,-0.0705
373,Hanna_2011,Adventure,207,0.1135,-0.1550,-0.0675
404,House of 1000 Corpses_2003,Horror,129,0.1029,0.0634,-0.1430
580,Next_2007,Sci-Fi,75,0.0824,0.1454,-0.0295
794,The Accountant_2016,Action,159,0.1105,-0.0617,-0.1955


**Figure.** Distributions of D1 and D2 in the primary sample.

In [ ]:
step.figures(res)["descriptors"].show()

**Table.** Descriptor quantiles in the primary sample.

In [ ]:
display(step.tables(res)["distribution"])

,descriptor (primary 834),count,mean,std,min,10%,25%,50%,75%,90%,max
0,D1,834.0,0.1297,0.0355,0.0553,0.0910,0.1052,0.1254,0.1473,0.1757,0.3187
1,D2,834.0,0.0097,0.1588,-0.5477,-0.1851,-0.0936,0.0143,0.1158,0.2020,0.6116
2,mean_level,834.0,-0.0173,0.1039,-0.3659,-0.1543,-0.0839,-0.0209,0.0578,0.1166,0.2759
3,min_position,834.0,0.4730,0.2951,0.0050,0.0750,0.2050,0.4750,0.7250,0.8750,0.9950
4,sign_changes,834.0,10.4436,3.8504,1.0000,6.0000,8.0000,10.0000,13.0000,16.0000,25.0000


**Table.** Spearman correlation of each descriptor with script size (primary sample).

In [ ]:
display(step.tables(res)["proxies"])

against,descriptor,mean sentences per scene,scenes on the timeline,sentences
0,D1,0.160,-0.299,-0.271
1,D2,0.023,0.015,0.059
2,mean_level,0.065,-0.039,0.067


**Figure.** D1 against mean scene length: the scene-noise effect that the scene-length control addresses (the Spearman correlation is in the proxies table).

In [ ]:
step.figures(res)["noise"].show()

**Table.** Split-half reliability: odd against even scenes, Spearman correlation across films and its Spearman-Brown correction to full length.

In [ ]:
display(step.tables(res)["reliability"])

,descriptor,split-half Spearman,Spearman-Brown
0,D1,0.270,0.425
1,D2,0.292,0.452
2,mean_level,0.741,0.851


**Figure.** D1 computed on odd scenes against D1 on even scenes.

In [ ]:
step.figures(res)["split_half"].show()

**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

Every one of the 1,087 scripts has an arc (834 in the primary sample). The median primary script has 133 scenes on its timeline, about 1.3 scenes per grid point. The average smoothed arc stays close to zero: -0.012 over the first 20%, -0.003 over the last 20%, and between -0.033 and +0.062 overall. Individual arcs swing much more than the average, which mostly cancels out (median D1 0.125, median D2 +0.014, 47% of films end lower than they start). The scores are DistilBERT SST-2 and NOT YET VALIDATED. **Proxies.** The strongest size correlate of D1 is scenes on the timeline (ρ = -0.299); of D2, sentences (ρ = +0.059); of mean level, sentences (ρ = +0.067). All are weak (|ρ| < 0.3), so the descriptors are not script size in disguise. The size controls of Step 7 still apply. **Reliability.** Split-half (odd vs even scenes, Spearman-Brown corrected): D1 0.43, D2 0.45, mean level 0.85. A reliability of r means roughly a share 1 − r of the between-film variance is scene-level noise: 57% for D1 and 55% for D2, against 15% for the mean level. That noise caps how strongly a descriptor can correlate with anything, profit included. Step 6 checks whether the order of scenes carries any structure at all.